In [1]:
# =====================================================================================================================================
#   Load Datasets  ( employees, rooms, room_booking_requests, room_booking  )
# =====================================================================================================================================


import pandas as pd
import numpy as np

employees = pd.read_csv("../data/raw/employees.csv")
rooms = pd.read_csv("../data/raw/rooms.csv")

room_booking_requests = pd.read_csv(
    "../data/raw/room_booking_requests.csv",
    parse_dates=[
        "requested_start_datetime",
        "requested_end_datetime"
    ]
)

room_bookings = pd.read_csv(
    "../data/raw/room_bookings.csv",
    parse_dates=[
        "requested_start_datetime",
        "requested_end_datetime",
        "allocated_start_datetime",
        "allocated_end_datetime"
    ]
)

print("Employees:", employees.shape)
print("Rooms:", rooms.shape)
print("Booking requests:", room_booking_requests.shape)
print("Room bookings:", room_bookings.shape)

Employees: (1000, 10)
Rooms: (50, 5)
Booking requests: (40000, 10)
Room bookings: (40000, 19)


In [2]:
# ======================================================================================================================
# CANDIDATE GENERATION KA EK HELPER
# ======================================================================================================================
def generate_allowed_start_times(requested_start, flexibility_minutes):
    allowed_times = []

    for offset in range(0, flexibility_minutes + 1, 15):
        if offset == 0:
            allowed_times.append(requested_start)
        else:
            earlier = requested_start - pd.Timedelta(minutes=offset)
            later = requested_start + pd.Timedelta(minutes=offset)

            allowed_times.append(earlier)
            allowed_times.append(later)

    return sorted(allowed_times)


print("Allowed start-time generator ready.")

Allowed start-time generator ready.


In [3]:
# ============================================================
# DETERMINISTIC FEASIBLE CANDIDATE GENERATOR
# ============================================================

similar_room_types = {
    "Conference": ["Meeting", "Training", "Focus"],
    "Meeting": ["Conference", "Training", "Focus"],
    "Training": ["Conference", "Meeting", "Focus"],
    "Focus": ["Meeting", "Conference", "Training"]
}

room_lookup = rooms.set_index("room_id").to_dict("index")

floors = sorted(rooms["floor"].unique())


def get_floor_order(requested_floor):
    return sorted(
        floors,
        key=lambda floor: (
            abs(floor - requested_floor),
            floor
        )
    )


def candidate_is_available(
    room_id,
    candidate_start,
    candidate_end,
    occupied_intervals
):
    """
    Check availability using ONLY bookings that already existed
    before the current request.
    """

    intervals = occupied_intervals.get(room_id, [])

    for existing_start, existing_end in intervals:

        if (
            candidate_start < existing_end
            and candidate_end > existing_start
        ):
            return False

    return True


def get_hierarchy_level(
    room,
    requested_type,
    requested_floor
):
    """
    Return the deterministic allocation hierarchy level
    that this room belongs to.
    """

    room_type = room["room_type"]
    room_floor = room["floor"]

    # Rule 1
    if (
        room_type == requested_type
        and room_floor == requested_floor
    ):
        return "exact_type_exact_floor"

    # Rule 2
    if (
        room_type == requested_type
        and abs(room_floor - requested_floor) == 1
    ):
        return "exact_type_nearby_floor"

    # Rule 3
    if room_type == requested_type:
        return "exact_type_any_floor"

    # Rule 4
    if (
        room_type in similar_room_types[requested_type]
        and room_floor == requested_floor
    ):
        return "similar_type_exact_floor"

    # Rule 5
    if room_type in similar_room_types[requested_type]:
        return "similar_type_any_floor"

    return None


def generate_feasible_candidates(
    request,
    occupied_intervals
):
    """
    Generate the candidates that the deterministic allocator
    would consider feasible for this request.

    Capacity is a hard constraint.

    At each candidate time:
        Rule 1 → Rule 2 → Rule 3 → Rule 4 → Rule 5

    Time is relaxed only if all five hierarchy levels fail.
    """

    requested_start = pd.Timestamp(
        request["requested_start_datetime"]
    )

    duration = pd.Timedelta(
        minutes=int(request["requested_duration_minutes"])
    )

    requested_end = requested_start + duration

    requested_type = request["requested_room_type"]
    requested_floor = int(request["requested_floor"])
    requested_capacity = int(request["requested_capacity"])
    flexibility = int(request["max_time_flexibility_minutes"])

    allowed_times = generate_allowed_start_times(
        requested_start,
        flexibility
    )

    # Requested time first, then closest ±15, ±30, ...
    allowed_times = sorted(
        allowed_times,
        key=lambda t: (
            abs((t - requested_start).total_seconds()),
            t
        )
    )

    hierarchy_order = [
        "exact_type_exact_floor",
        "exact_type_nearby_floor",
        "exact_type_any_floor",
        "similar_type_exact_floor",
        "similar_type_any_floor"
    ]

    # --------------------------------------------------------
    # Check one time level at a time
    # --------------------------------------------------------

    for candidate_start in allowed_times:

        candidate_end = candidate_start + duration

        feasible_by_rule = {
            rule: []
            for rule in hierarchy_order
        }

        for room_id, room in room_lookup.items():

            # Capacity = HARD constraint
            if room["capacity"] < requested_capacity:
                continue

            rule = get_hierarchy_level(
                room,
                requested_type,
                requested_floor
            )

            if rule is None:
                continue

            if candidate_is_available(
                room_id,
                candidate_start,
                candidate_end,
                occupied_intervals
            ):
                feasible_by_rule[rule].append(
                    (
                        room_id,
                        room,
                        candidate_start,
                        candidate_end
                    )
                )

        # ----------------------------------------------------
        # First hierarchy level containing feasible candidates
        # ----------------------------------------------------

        for rule in hierarchy_order:

            candidates = feasible_by_rule[rule]

            if candidates:

                candidate_rows = []

                for (
                    room_id,
                    room,
                    start,
                    end
                ) in candidates:

                    candidate_rows.append({

                        "booking_id":
                            request["booking_id"],

                        "employee_id":
                            request["employee_id"],

                        "candidate_room_id":
                            room_id,

                        "candidate_room_type":
                            room["room_type"],

                        "candidate_floor":
                            room["floor"],

                        "candidate_capacity":
                            room["capacity"],

                        "candidate_start_datetime":
                            start,

                        "candidate_end_datetime":
                            end,

                        "allocation_rule":
                            rule,

                        "time_relaxation_minutes":
                            abs(
                                int(
                                    (
                                        start
                                        - requested_start
                                    ).total_seconds()
                                    / 60
                                )
                            )
                    })

                return pd.DataFrame(candidate_rows)

    # No feasible candidate anywhere within flexibility
    return pd.DataFrame()

In [4]:
# ============================================================
# TEST CANDIDATE GENERATION ON ONE HISTORICAL REQUEST
# ============================================================

# Requests ko allocator ke original processing order mein rakho
requests_ordered = (
    room_booking_requests
    .sort_values("booking_id")
    .reset_index(drop=True)
)

# Ek allocated historical request select karo
test_request = requests_ordered.iloc[100]

test_booking_id = test_request["booking_id"]

# Current request se PEHLE process hui allocated bookings
previous_bookings = (
    room_bookings[
        (room_bookings["booking_status"] == "Allocated") &
        (room_bookings["booking_id"] < test_booking_id)
    ]
    .sort_values("allocated_start_datetime")
)

# Room-wise occupied intervals
occupied_intervals = {}

for room_id, group in previous_bookings.groupby("allocated_room_id"):

    occupied_intervals[room_id] = list(
        zip(
            group["allocated_start_datetime"],
            group["allocated_end_datetime"]
        )
    )

# Generate feasible candidates
test_candidates = generate_feasible_candidates(
    test_request,
    occupied_intervals
)

print("Test booking:", test_booking_id)
print("Employee:", test_request["employee_id"])
print("Requested room type:", test_request["requested_room_type"])
print("Requested floor:", test_request["requested_floor"])
print("Requested capacity:", test_request["requested_capacity"])
print("Requested start:", test_request["requested_start_datetime"])
print("Flexibility:", test_request["max_time_flexibility_minutes"])

print("\nNumber of feasible candidates:", len(test_candidates))

test_candidates

Test booking: RB00101
Employee: E0405
Requested room type: Training
Requested floor: 2
Requested capacity: 6
Requested start: 2025-12-25 09:00:00
Flexibility: 30

Number of feasible candidates: 1


,booking_id,employee_id,candidate_room_id,candidate_room_type,candidate_floor,candidate_capacity,candidate_start_datetime,candidate_end_datetime,allocation_rule,time_relaxation_minutes
0,RB00101,E0405,R018,Training,2,16,2025-12-25 09:00:00,2025-12-25 10:30:00,exact_type_exact_floor,0


In [5]:
# ============================================================
# CELL 5 — BUILD FULL FEASIBLE CANDIDATE DATASET
# ============================================================

# Same chronological order used by the original allocator
requests_ordered = (
    room_booking_requests
    .sort_values(
        [
            "booking_date",
            "requested_start_datetime",
            "booking_id"
        ]
    )
    .reset_index(drop=True)
)

# Historical allocation lookup
historical_allocations = (
    room_bookings[
        room_bookings["booking_status"] == "Allocated"
    ]
    .copy()
)

allocation_lookup = (
    historical_allocations
    .set_index("booking_id")
    .to_dict("index")
)

# Occupancy state at the current point in time
occupied_intervals = {
    room_id: []
    for room_id in rooms["room_id"]
}

all_candidate_rows = []

# ------------------------------------------------------------
# Replay requests chronologically
# ------------------------------------------------------------

for i, request in requests_ordered.iterrows():

    booking_id = request["booking_id"]

    # --------------------------------------------------------
    # 1. Find ALL feasible candidates using only bookings
    #    that have already occurred in the replay
    # --------------------------------------------------------

    candidates = generate_feasible_candidates(
        request,
        occupied_intervals
    )

    if not candidates.empty:
        all_candidate_rows.append(candidates)

    # --------------------------------------------------------
    # 2. Add the HISTORICAL allocation of this request
    #    to occupancy before moving to the next request
    # --------------------------------------------------------

    if booking_id in allocation_lookup:

        allocation = allocation_lookup[booking_id]

        room_id = allocation["allocated_room_id"]
        start = allocation["allocated_start_datetime"]
        end = allocation["allocated_end_datetime"]

        occupied_intervals[room_id].append(
            (start, end)
        )

        # Keep intervals sorted for future availability checks
        occupied_intervals[room_id].sort()

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (i + 1) % 5000 == 0:
        print(
            f"Processed {i + 1:,} / "
            f"{len(requests_ordered):,} requests"
        )


# ------------------------------------------------------------
# Combine candidate rows
# ------------------------------------------------------------

if all_candidate_rows:

    candidate_dataset = pd.concat(
        all_candidate_rows,
        ignore_index=True
    )

else:

    candidate_dataset = pd.DataFrame()


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nCandidate dataset created.")

print(
    "Total candidate rows:",
    len(candidate_dataset)
)

print(
    "Unique requests with candidates:",
    candidate_dataset["booking_id"].nunique()
)

print(
    "Unique employees:",
    candidate_dataset["employee_id"].nunique()
)

print("\nCandidate count distribution:")

print(
    candidate_dataset
    .groupby("booking_id")
    .size()
    .value_counts()
    .sort_index()
)

print("\nFirst 5 candidate rows:")

candidate_dataset.head()

Processed 5,000 / 40,000 requests
Processed 10,000 / 40,000 requests
Processed 15,000 / 40,000 requests
Processed 20,000 / 40,000 requests
Processed 25,000 / 40,000 requests
Processed 30,000 / 40,000 requests
Processed 35,000 / 40,000 requests
Processed 40,000 / 40,000 requests

Candidate dataset created.
Total candidate rows: 61784
Unique requests with candidates: 32442
Unique employees: 998

Candidate count distribution:
1     18579
2      6932
3      3426
4      1613
5       823
6       405
7       222
8       131
9        90
10       67
11       42
12       45
13       24
14       17
15        9
16        6
17        4
18        3
19        3
22        1
Name: count, dtype: int64

First 5 candidate rows:


,booking_id,employee_id,candidate_room_id,candidate_room_type,candidate_floor,candidate_capacity,candidate_start_datetime,candidate_end_datetime,allocation_rule,time_relaxation_minutes
0,RB06430,E0742,R008,Meeting,1,16,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_any_floor,0
1,RB10999,E0166,R042,Conference,5,4,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_exact_floor,0
2,RB10999,E0166,R045,Conference,5,8,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_exact_floor,0
3,RB19847,E0605,R034,Conference,4,16,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_any_floor,0
4,RB22582,E0904,R024,Meeting,3,8,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_exact_floor,0


In [6]:
# ============================================================
# CELL 6 — LOAD BEHAVIOUR FEATURES
# ============================================================

room_type_behavior_features = pd.read_csv(
    "../data/processed/room_type_behavior_features.csv"
)

floor_behavior_features = pd.read_csv(
    "../data/processed/floor_behavior_features.csv"
)

capacity_behavior_features = pd.read_csv(
    "../data/processed/capacity_behavior_features.csv"
)

start_hour_behavior_features = pd.read_csv(
    "../data/processed/start_hour_behavior_features.csv"
)

print("Room type:", room_type_behavior_features.shape)
print("Floor:", floor_behavior_features.shape)
print("Capacity:", capacity_behavior_features.shape)
print("Start hour:", start_hour_behavior_features.shape)

print("\nRoom type columns:")
print(room_type_behavior_features.columns.tolist())

print("\nFloor columns:")
print(floor_behavior_features.columns.tolist())

print("\nCapacity columns:")
print(capacity_behavior_features.columns.tolist())

print("\nStart-hour columns:")
print(start_hour_behavior_features.columns.tolist())

Room type: (880, 10)
Floor: (880, 12)
Capacity: (880, 16)
Start hour: (880, 10)

Room type columns:
['employee_id', 'Conference_long_term', 'Focus_long_term', 'Meeting_long_term', 'Training_long_term', 'Conference_recent', 'Focus_recent', 'Meeting_recent', 'Training_recent', 'room_type_shift_pct']

Floor columns:
['employee_id', 'floor_1_pct_long_term', 'floor_2_pct_long_term', 'floor_3_pct_long_term', 'floor_4_pct_long_term', 'floor_5_pct_long_term', 'floor_1_pct_recent', 'floor_2_pct_recent', 'floor_3_pct_recent', 'floor_4_pct_recent', 'floor_5_pct_recent', 'floor_shift_pct']

Capacity columns:
['employee_id', 'capacity_2_pct_long_term', 'capacity_4_pct_long_term', 'capacity_6_pct_long_term', 'capacity_8_pct_long_term', 'capacity_10_pct_long_term', 'capacity_12_pct_long_term', 'capacity_16_pct_long_term', 'capacity_2_pct_recent', 'capacity_4_pct_recent', 'capacity_6_pct_recent', 'capacity_8_pct_recent', 'capacity_10_pct_recent', 'capacity_12_pct_recent', 'capacity_16_pct_recent', 'ca

In [7]:
# ============================================================
# CELL 7 — MERGE BEHAVIOUR FEATURES INTO CANDIDATE DATASET
# ============================================================

candidate_dataset = (
    candidate_dataset
    .merge(
        room_type_behavior_features,
        on="employee_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        floor_behavior_features,
        on="employee_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        capacity_behavior_features,
        on="employee_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        start_hour_behavior_features,
        on="employee_id",
        how="left",
        validate="many_to_one"
    )
)

print("Candidate dataset shape:", candidate_dataset.shape)
print("Unique bookings:", candidate_dataset["booking_id"].nunique())
print("Unique employees:", candidate_dataset["employee_id"].nunique())

print("\nTotal missing values:")
print(candidate_dataset.isna().sum().sum())

print("\nMissing values by feature group:")
print(
    candidate_dataset[
        [
            "room_type_shift_pct",
            "floor_shift_pct",
            "capacity_shift_pct",
            "start_hour_shift_pct"
        ]
    ].isna().sum()
)

print("\nCandidate dataset preview:")
candidate_dataset.head()

Candidate dataset shape: (61784, 54)
Unique bookings: 32442
Unique employees: 998

Total missing values:
134376

Missing values by feature group:
room_type_shift_pct     3054
floor_shift_pct         3054
capacity_shift_pct      3054
start_hour_shift_pct    3054
dtype: int64

Candidate dataset preview:


,booking_id,employee_id,candidate_room_id,candidate_room_type,candidate_floor,candidate_capacity,candidate_start_datetime,candidate_end_datetime,allocation_rule,time_relaxation_minutes,...,capacity_shift_pct,start_hour_8_pct_long_term,start_hour_9_pct_long_term,start_hour_10_pct_long_term,start_hour_11_pct_long_term,start_hour_8_pct_recent,start_hour_9_pct_recent,start_hour_10_pct_recent,start_hour_11_pct_recent,start_hour_shift_pct
0,RB06430,E0742,R008,Meeting,1,16,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_any_floor,0,...,22.857143,20.000000,54.285714,25.714286,0.0,33.333333,66.666667,0.0,0.0,25.714286
1,RB10999,E0166,R042,Conference,5,4,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_exact_floor,0,...,23.589744,28.205128,64.102564,7.692308,0.0,0.000000,60.000000,40.0,0.0,32.307692
2,RB10999,E0166,R045,Conference,5,8,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_exact_floor,0,...,23.589744,28.205128,64.102564,7.692308,0.0,0.000000,60.000000,40.0,0.0,32.307692
3,RB19847,E0605,R034,Conference,4,16,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_any_floor,0,...,19.047619,14.285714,69.047619,16.666667,0.0,0.000000,100.000000,0.0,0.0,30.952381
4,RB22582,E0904,R024,Meeting,3,8,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_exact_floor,0,...,17.567568,78.378378,21.621622,0.000000,0.0,100.000000,0.000000,0.0,0.0,21.621622


In [8]:
# ============================================================
# CELL 8 — CHECK EMPLOYEES MISSING BEHAVIOUR PROFILES
# ============================================================

candidate_employees = set(
    candidate_dataset["employee_id"].unique()
)

behaviour_employees = set(
    room_type_behavior_features["employee_id"].unique()
)

missing_behaviour_employees = sorted(
    candidate_employees - behaviour_employees
)

print(
    "Employees with candidate rows:",
    len(candidate_employees)
)

print(
    "Employees with behaviour profiles:",
    len(behaviour_employees)
)

print(
    "Employees missing behaviour profiles:",
    len(missing_behaviour_employees)
)

print("\nMissing employee IDs:")
print(missing_behaviour_employees)

Employees with candidate rows: 998
Employees with behaviour profiles: 880
Employees missing behaviour profiles: 118

Missing employee IDs:
['E0016', 'E0022', 'E0028', 'E0043', 'E0057', 'E0068', 'E0073', 'E0076', 'E0085', 'E0089', 'E0099', 'E0103', 'E0106', 'E0107', 'E0112', 'E0133', 'E0143', 'E0145', 'E0153', 'E0173', 'E0178', 'E0179', 'E0204', 'E0227', 'E0229', 'E0234', 'E0238', 'E0241', 'E0242', 'E0265', 'E0271', 'E0280', 'E0281', 'E0286', 'E0325', 'E0342', 'E0345', 'E0365', 'E0376', 'E0377', 'E0390', 'E0410', 'E0424', 'E0426', 'E0439', 'E0449', 'E0450', 'E0451', 'E0454', 'E0462', 'E0476', 'E0480', 'E0489', 'E0502', 'E0511', 'E0512', 'E0529', 'E0530', 'E0532', 'E0533', 'E0540', 'E0549', 'E0564', 'E0577', 'E0586', 'E0596', 'E0608', 'E0611', 'E0612', 'E0631', 'E0648', 'E0651', 'E0656', 'E0658', 'E0666', 'E0668', 'E0676', 'E0680', 'E0712', 'E0716', 'E0739', 'E0752', 'E0761', 'E0762', 'E0770', 'E0773', 'E0774', 'E0778', 'E0784', 'E0795', 'E0798', 'E0800', 'E0802', 'E0803', 'E0810', 'E083

In [9]:
# ============================================================
# CELL 8 — EMPLOYEE ↔ CANDIDATE HISTORICAL RELATIONSHIP
# ============================================================

from collections import defaultdict

# Requests in the same chronological order as the allocator
requests_ordered = (
    room_booking_requests
    .sort_values(
        ["booking_date", "requested_start_datetime", "booking_id"]
    )
    .reset_index(drop=True)
)

# Historical allocated bookings
historical_allocations = (
    room_bookings[
        room_bookings["booking_status"] == "Allocated"
    ]
    .copy()
)

allocation_lookup = (
    historical_allocations
    .set_index("booking_id")
    .to_dict("index")
)

# ------------------------------------------------------------
# Historical behaviour state
# ------------------------------------------------------------

employee_total = defaultdict(int)

employee_room_count = defaultdict(lambda: defaultdict(int))
employee_type_count = defaultdict(lambda: defaultdict(int))
employee_floor_count = defaultdict(lambda: defaultdict(int))
employee_hour_count = defaultdict(lambda: defaultdict(int))

relationship_rows = []


# ------------------------------------------------------------
# Replay requests chronologically
# ------------------------------------------------------------

for _, request in requests_ordered.iterrows():

    booking_id = request["booking_id"]
    employee_id = request["employee_id"]

    # --------------------------------------------------------
    # Candidates belonging to the current request
    # --------------------------------------------------------

    current_candidates = candidate_dataset[
        candidate_dataset["booking_id"] == booking_id
    ]

    if not current_candidates.empty:

        total_history = employee_total[employee_id]

        for _, candidate in current_candidates.iterrows():

            room_id = candidate["candidate_room_id"]
            room_type = candidate["candidate_room_type"]
            floor = candidate["candidate_floor"]

            candidate_start = pd.Timestamp(
                candidate["candidate_start_datetime"]
            )

            candidate_hour = candidate_start.hour

            # ------------------------------------------------
            # Room-specific relationship
            # ------------------------------------------------

            room_count = employee_room_count[
                employee_id
            ][room_id]

            room_pct = (
                room_count / total_history
                if total_history > 0
                else 0.0
            )

            # ------------------------------------------------
            # Room-type relationship
            # ------------------------------------------------

            type_count = employee_type_count[
                employee_id
            ][room_type]

            type_pct = (
                type_count / total_history
                if total_history > 0
                else 0.0
            )

            # ------------------------------------------------
            # Floor relationship
            # ------------------------------------------------

            floor_count = employee_floor_count[
                employee_id
            ][floor]

            floor_pct = (
                floor_count / total_history
                if total_history > 0
                else 0.0
            )

            # ------------------------------------------------
            # Start-hour relationship
            # ------------------------------------------------

            hour_count = employee_hour_count[
                employee_id
            ][candidate_hour]

            hour_pct = (
                hour_count / total_history
                if total_history > 0
                else 0.0
            )

            relationship_rows.append({

                "booking_id": booking_id,
                "employee_id": employee_id,
                "candidate_room_id": room_id,

                "employee_history_count":
                    total_history,

                "has_employee_history":
                    int(total_history > 0),

                "employee_candidate_room_count":
                    room_count,

                "employee_candidate_room_pct":
                    room_pct,

                "employee_candidate_type_count":
                    type_count,

                "employee_candidate_type_pct":
                    type_pct,

                "employee_candidate_floor_count":
                    floor_count,

                "employee_candidate_floor_pct":
                    floor_pct,

                "employee_candidate_hour_count":
                    hour_count,

                "employee_candidate_hour_pct":
                    hour_pct
            })

    # --------------------------------------------------------
    # AFTER evaluating current request:
    # add its HISTORICAL allocation to employee history
    # --------------------------------------------------------

    if booking_id in allocation_lookup:

        allocation = allocation_lookup[booking_id]

        allocated_room = allocation["allocated_room_id"]
        allocated_start = pd.Timestamp(
            allocation["allocated_start_datetime"]
        )

        # Room information
        allocated_room_info = room_lookup[allocated_room]

        allocated_type = allocated_room_info["room_type"]
        allocated_floor = allocated_room_info["floor"]
        allocated_hour = allocated_start.hour

        # Update employee history
        employee_total[employee_id] += 1

        employee_room_count[
            employee_id
        ][allocated_room] += 1

        employee_type_count[
            employee_id
        ][allocated_type] += 1

        employee_floor_count[
            employee_id
        ][allocated_floor] += 1

        employee_hour_count[
            employee_id
        ][allocated_hour] += 1


# ------------------------------------------------------------
# Create relationship feature dataframe
# ------------------------------------------------------------

candidate_relationship_features = pd.DataFrame(
    relationship_rows
)

print(
    "Relationship feature rows:",
    len(candidate_relationship_features)
)

print(
    "Unique bookings:",
    candidate_relationship_features["booking_id"].nunique()
)

print(
    "Unique employees:",
    candidate_relationship_features["employee_id"].nunique()
)

print("\nRelationship features created:")
print(candidate_relationship_features.columns.tolist())

candidate_relationship_features.head()

Relationship feature rows: 61784
Unique bookings: 32442
Unique employees: 998

Relationship features created:
['booking_id', 'employee_id', 'candidate_room_id', 'employee_history_count', 'has_employee_history', 'employee_candidate_room_count', 'employee_candidate_room_pct', 'employee_candidate_type_count', 'employee_candidate_type_pct', 'employee_candidate_floor_count', 'employee_candidate_floor_pct', 'employee_candidate_hour_count', 'employee_candidate_hour_pct']


,booking_id,employee_id,candidate_room_id,employee_history_count,has_employee_history,employee_candidate_room_count,employee_candidate_room_pct,employee_candidate_type_count,employee_candidate_type_pct,employee_candidate_floor_count,employee_candidate_floor_pct,employee_candidate_hour_count,employee_candidate_hour_pct
0,RB06430,E0742,R008,0,0,0,0.0,0,0.0,0,0.0,0,0.0
1,RB10999,E0166,R042,0,0,0,0.0,0,0.0,0,0.0,0,0.0
2,RB10999,E0166,R045,0,0,0,0.0,0,0.0,0,0.0,0,0.0
3,RB19847,E0605,R034,0,0,0,0.0,0,0.0,0,0.0,0,0.0
4,RB22582,E0904,R024,0,0,0,0.0,0,0.0,0,0.0,0,0.0


In [12]:
# =====================================================================================================================
# Merge relationship features -  [ employee profile + candidate info + employee ↔ candidate history ]
# =====================================================================================================================

candidate_dataset = candidate_dataset.merge(
    candidate_relationship_features,
    on=[
        "booking_id",
        "employee_id",
        "candidate_room_id"
    ],
    how="left",
    validate="one_to_one"
)

print("Candidate dataset shape:", candidate_dataset.shape)
print("Total missing values:", candidate_dataset.isna().sum().sum())

candidate_dataset.head()

Candidate dataset shape: (61784, 84)
Total missing values: 134376


,booking_id,employee_id,candidate_room_id,candidate_room_type,candidate_floor,candidate_capacity,candidate_start_datetime,candidate_end_datetime,allocation_rule,time_relaxation_minutes,...,employee_history_count,has_employee_history,employee_candidate_room_count,employee_candidate_room_pct,employee_candidate_type_count,employee_candidate_type_pct,employee_candidate_floor_count,employee_candidate_floor_pct,employee_candidate_hour_count,employee_candidate_hour_pct
0,RB06430,E0742,R008,Meeting,1,16,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_any_floor,0,...,0,0,0,0.0,0,0.0,0,0.0,0,0.0
1,RB10999,E0166,R042,Conference,5,4,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_exact_floor,0,...,0,0,0,0.0,0,0.0,0,0.0,0,0.0
2,RB10999,E0166,R045,Conference,5,8,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_exact_floor,0,...,0,0,0,0.0,0,0.0,0,0.0,0,0.0
3,RB19847,E0605,R034,Conference,4,16,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_any_floor,0,...,0,0,0,0.0,0,0.0,0,0.0,0,0.0
4,RB22582,E0904,R024,Meeting,3,8,2025-01-01 08:00:00,2025-01-01 09:15:00,exact_type_exact_floor,0,...,0,0,0,0.0,0,0.0,0,0.0,0,0.0


In [19]:
# ===================================================================================================================================
#                                                Introducing - SCARCITY FEATURES    
# ====================================================================================================================================


# 1.---------------------------------------------------   [ Room Historical Demand ]    -------------------------------------------------

# Create chronological request sequence
request_sequence = (
    requests_ordered[["booking_id"]]
    .reset_index(drop=True)
    .reset_index()
    .rename(columns={"index": "request_sequence"})
)

# Add request sequence to candidate rows
candidate_with_sequence = candidate_dataset.merge(
    request_sequence,
    on="booking_id",
    how="left",
    validate="many_to_one"
)

# Historical allocated bookings with chronological sequence
allocation_events = (
    historical_allocations
    .merge(
        request_sequence,
        on="booking_id",
        how="left",
        validate="one_to_one"
    )
    .sort_values("request_sequence")
    .reset_index(drop=True)
)

# Count previous allocations for each room
allocation_events["room_historical_demand"] = (
    allocation_events
    .groupby("allocated_room_id")
    .cumcount()
)

# Rename room column for candidate matching
demand_events = (
    allocation_events[
        [
            "allocated_room_id",
            "request_sequence",
            "room_historical_demand"
        ]
    ]
    .rename(
        columns={
            "allocated_room_id": "candidate_room_id"
        }
    )
)

# IMPORTANT:
# merge_asof requires the 'on' key to be globally sorted
candidate_with_sequence = candidate_with_sequence.sort_values(
    ["request_sequence", "candidate_room_id"]
).reset_index(drop=True)

demand_events = demand_events.sort_values(
    ["request_sequence", "candidate_room_id"]
).reset_index(drop=True)

# Get latest historical demand before current request
candidate_with_demand = pd.merge_asof(
    candidate_with_sequence,
    demand_events,
    on="request_sequence",
    by="candidate_room_id",
    direction="backward",
    allow_exact_matches=False
)

# No previous booking = zero historical demand
candidate_with_demand["room_historical_demand"] = (
    candidate_with_demand["room_historical_demand"]
    .fillna(0)
    .astype(int)
)

# Restore original candidate order
candidate_dataset = (
    candidate_with_demand
    .sort_index()
    .drop(columns=["request_sequence"])
)

print(
    "Candidate dataset shape:",
    candidate_dataset.shape
)

print(
    candidate_dataset[
        [
            "booking_id",
            "candidate_room_id",
            "room_historical_demand"
        ]
    ].tail(10)
)

Candidate dataset shape: (61784, 86)
      booking_id candidate_room_id  room_historical_demand
61774    RB27197              R016                     454
61775    RB27197              R019                     352
61776    RB33422              R031                     643
61777    RB33422              R036                     366
61778    RB35720              R019                     352
61779    RB36542              R040                     922
61780    RB24451              R041                     939
61781    RB34216              R006                     948
61782    RB01528              R024                     849
61783    RB01528              R025                     779


In [ ]:
# 2.--------------------------------------------   [Time Based Room Historical Demand ]    -------------------------------------------------

from collections import defaultdict
import numpy as np

# --------------------------------------------------
# 1. Chronological request sequence
# --------------------------------------------------

requests_ordered = (
    room_booking_requests
    .sort_values(
        [
            "booking_date",
            "requested_start_datetime",
            "booking_id"
        ]
    )
    .reset_index(drop=True)
)

request_sequence = (
    requests_ordered[["booking_id"]]
    .reset_index()
    .rename(columns={"index": "request_sequence"})
)


# --------------------------------------------------
# 2. Add chronological sequence to historical allocations
# --------------------------------------------------

historical_allocations = (
    room_bookings[
        room_bookings["booking_status"] == "Allocated"
    ]
    .copy()
)

allocation_events = (
    historical_allocations
    .merge(
        request_sequence,
        on="booking_id",
        how="left",
        validate="one_to_one"
    )
)

allocation_events["allocated_hour"] = (
    pd.to_datetime(
        allocation_events["allocated_start_datetime"]
    ).dt.hour
)


# --------------------------------------------------
# 3. Build fast lookup:
#    (room, hour) -> sorted historical request sequences
# --------------------------------------------------

room_hour_history = defaultdict(list)

for row in allocation_events.itertuples():

    key = (
        row.allocated_room_id,
        row.allocated_hour
    )

    room_hour_history[key].append(
        row.request_sequence
    )

for key in room_hour_history:
    room_hour_history[key].sort()


# --------------------------------------------------
# 4. Add request sequence + candidate hour
# --------------------------------------------------

candidate_work = candidate_dataset.merge(
    request_sequence,
    on="booking_id",
    how="left",
    validate="many_to_one"
)

candidate_work["candidate_hour"] = (
    pd.to_datetime(
        candidate_work["candidate_start_datetime"]
    ).dt.hour
)


# --------------------------------------------------
# 5. Fast historical demand lookup
# --------------------------------------------------

candidate_work["room_time_demand"] = [
    np.searchsorted(
        room_hour_history.get(
            (room_id, hour),
            []
        ),
        sequence,
        side="left"
    )
    for room_id, hour, sequence
    in zip(
        candidate_work["candidate_room_id"],
        candidate_work["candidate_hour"],
        candidate_work["request_sequence"]
    )
]


# --------------------------------------------------
# 6. Update candidate dataset
# --------------------------------------------------

candidate_dataset = (
    candidate_work
    .drop(
        columns=[
            "request_sequence",
            "candidate_hour"
        ]
    )
)

print(
    "Candidate dataset shape:",
    candidate_dataset.shape
)

print(
    candidate_dataset[
        [
            "booking_id",
            "candidate_room_id",
            "candidate_start_datetime",
            "room_historical_demand",
            "room_time_demand"
        ]
    ].tail(10)
)

Candidate dataset shape: (61784, 86)
      booking_id candidate_room_id candidate_start_datetime  \
61774    RB27197              R016      2025-12-31 10:45:00   
61775    RB27197              R019      2025-12-31 10:45:00   
61776    RB33422              R031      2025-12-31 10:45:00   
61777    RB33422              R036      2025-12-31 10:45:00   
61778    RB35720              R019      2025-12-31 10:45:00   
61779    RB36542              R040      2025-12-31 11:00:00   
61780    RB24451              R041      2025-12-31 11:30:00   
61781    RB34216              R006      2025-12-31 11:45:00   
61782    RB01528              R024      2025-12-31 11:30:00   
61783    RB01528              R025      2025-12-31 11:30:00   

       room_historical_demand  room_time_demand  
61774                     454               146  
61775                     352               118  
61776                     643               188  
61777                     366               125  
61778              

In [28]:
# 3.---------------------------------------------------   [ Room Utilization rate ]    -------------------------------------------------

# --------------------------------------------------
# 1. Add chronological request sequence
# --------------------------------------------------

candidate_work = candidate_dataset.merge(
    request_sequence,
    on="booking_id",
    how="left",
    validate="many_to_one"
)


# --------------------------------------------------
# 2. Historical allocation duration
# --------------------------------------------------

allocation_events["allocated_duration_minutes"] = (
    (
        pd.to_datetime(allocation_events["allocated_end_datetime"])
        - pd.to_datetime(allocation_events["allocated_start_datetime"])
    )
    .dt.total_seconds()
    / 60
)


# --------------------------------------------------
# 3. Cumulative occupied minutes for each room
# --------------------------------------------------

allocation_events = allocation_events.sort_values(
    "request_sequence"
).reset_index(drop=True)

allocation_events["room_cumulative_occupied_minutes"] = (
    allocation_events
    .groupby("allocated_room_id")[
        "allocated_duration_minutes"
    ]
    .cumsum()
)


# --------------------------------------------------
# 4. Build room history lookup
# --------------------------------------------------

room_utilization_history = defaultdict(list)

for row in allocation_events.itertuples():

    room_utilization_history[
        row.allocated_room_id
    ].append(
        (
            row.request_sequence,
            row.room_cumulative_occupied_minutes
        )
    )


# --------------------------------------------------
# 5. Calculate elapsed working time
# --------------------------------------------------

start_date = requests_ordered["booking_date"].min()
end_dates = pd.to_datetime(
    candidate_work["booking_id"].map(
        requests_ordered.set_index("booking_id")["booking_date"]
    )
)

business_days = (
    pd.bdate_range(
        start=start_date,
        end=end_dates.max()
    )
)

# Use the actual observed workplace request window
workday_start_hour = (
    pd.to_datetime(
        requests_ordered["requested_start_datetime"]
    ).dt.hour.min()
)

workday_end_hour = (
    pd.to_datetime(
        requests_ordered["requested_end_datetime"]
    ).dt.hour.max() + 1
)

working_minutes_per_day = (
    workday_end_hour - workday_start_hour
) * 60

business_day_number = (
    pd.to_datetime(
        candidate_work["booking_id"].map(
            requests_ordered.set_index("booking_id")["booking_date"]
        )
    )
    .dt.normalize()
    .map(
        {
            date: i + 1
            for i, date in enumerate(business_days)
        }
    )
)

elapsed_working_minutes = (
    (business_day_number - 1)
    * working_minutes_per_day
)

# Add time elapsed within current day
candidate_time = pd.to_datetime(
    candidate_work["candidate_start_datetime"]
)

elapsed_working_minutes += (
    (
        candidate_time.dt.hour - workday_start_hour
    ) * 60
    + candidate_time.dt.minute
)

elapsed_working_minutes = elapsed_working_minutes.clip(
    lower=1
)


# --------------------------------------------------
# 6. Fast room utilization lookup
# --------------------------------------------------

candidate_work["room_cumulative_occupied_minutes"] = [
    (
        room_utilization_history.get(room_id, [])
    )
    for room_id in candidate_work["candidate_room_id"]
]

candidate_work["room_cumulative_occupied_minutes"] = [
    values[
        np.searchsorted(
            [x[0] for x in values],
            sequence,
            side="left"
        ) - 1
    ][1]
    if values and np.searchsorted(
        [x[0] for x in values],
        sequence,
        side="left"
    ) > 0
    else 0.0
    for values, sequence
    in zip(
        candidate_work["room_cumulative_occupied_minutes"],
        candidate_work["request_sequence"]
    )
]


# --------------------------------------------------
# 7. Final utilization rate
# --------------------------------------------------

candidate_work["room_utilization_rate"] = (
    candidate_work["room_cumulative_occupied_minutes"]
    / elapsed_working_minutes
)

candidate_work["room_utilization_rate"] = (
    candidate_work["room_utilization_rate"]
    .clip(lower=0, upper=1)
)


# --------------------------------------------------
# 8. Update candidate dataset
# --------------------------------------------------

candidate_dataset = (
    candidate_work
    .drop(columns=["request_sequence"])
)

print(
    "Candidate dataset shape:",
    candidate_dataset.shape
)

print(
    candidate_dataset[
        [
            "booking_id",
            "candidate_room_id",
            "candidate_start_datetime",
            "room_historical_demand",
            "room_time_demand",
            "room_utilization_rate"
        ]
    ].tail(10)
)

Candidate dataset shape: (61784, 88)
      booking_id candidate_room_id candidate_start_datetime  \
61774    RB27197              R016      2025-12-31 10:45:00   
61775    RB27197              R019      2025-12-31 10:45:00   
61776    RB33422              R031      2025-12-31 10:45:00   
61777    RB33422              R036      2025-12-31 10:45:00   
61778    RB35720              R019      2025-12-31 10:45:00   
61779    RB36542              R040      2025-12-31 11:00:00   
61780    RB24451              R041      2025-12-31 11:30:00   
61781    RB34216              R006      2025-12-31 11:45:00   
61782    RB01528              R024      2025-12-31 11:30:00   
61783    RB01528              R025      2025-12-31 11:30:00   

       room_historical_demand  room_time_demand  room_utilization_rate  
61774                     454               146               0.327148  
61775                     352               118               0.253559  
61776                     643               188   

In [29]:
# 4.---------------------------------------------------   [ Capacity Feasible Count ]    -------------------------------------------------

# Capacity-feasible alternatives for each candidate
# Counts other rooms that satisfy capacity and are available
# at the candidate's exact start/end time.

room_ids = rooms["room_id"].tolist()

room_capacity = rooms.set_index("room_id")["capacity"].to_dict()

# Reconstruct historical occupancy chronologically
requests_ordered = (
    room_booking_requests
    .sort_values(["booking_date", "requested_start_datetime", "booking_id"])
    .reset_index(drop=True)
)

request_sequence = (
    requests_ordered[["booking_id"]]
    .reset_index()
    .rename(columns={"index": "request_sequence"})
)

allocated = (
    room_bookings[
        room_bookings["booking_status"] == "Allocated"
    ][
        [
            "booking_id",
            "allocated_room_id",
            "allocated_start_datetime",
            "allocated_end_datetime"
        ]
    ]
    .merge(
        request_sequence,
        on="booking_id",
        how="left",
        validate="one_to_one"
    )
    .sort_values("request_sequence")
)

# Occupied intervals available BEFORE each request
occupied_intervals = {room_id: [] for room_id in room_ids}

capacity_alternative_counts = []

for row in candidate_dataset.sort_values(
    ["booking_id", "candidate_start_datetime", "candidate_room_id"]
).itertuples():

    candidate_start = row.candidate_start_datetime
    candidate_end = row.candidate_end_datetime
    requested_capacity = row.candidate_capacity
    candidate_room = row.candidate_room_id

    count = 0

    for room_id in room_ids:

        # Don't count the candidate room itself
        if room_id == candidate_room:
            continue

        # Hard capacity constraint
        if room_capacity[room_id] < requested_capacity:
            continue

        # Check availability
        available = True

        for existing_start, existing_end in occupied_intervals[room_id]:
            if candidate_start < existing_end and candidate_end > existing_start:
                available = False
                break

        if available:
            count += 1

    capacity_alternative_counts.append(
        (
            row.booking_id,
            row.candidate_room_id,
            row.candidate_start_datetime,
            count
        )
    )

# Add the feature back to candidate_dataset
capacity_alternatives = pd.DataFrame(
    capacity_alternative_counts,
    columns=[
        "booking_id",
        "candidate_room_id",
        "candidate_start_datetime",
        "capacity_feasible_alternative_count"
    ]
)

candidate_dataset = candidate_dataset.merge(
    capacity_alternatives,
    on=[
        "booking_id",
        "candidate_room_id",
        "candidate_start_datetime"
    ],
    how="left",
    validate="one_to_one"
)

# Now replay historical allocations so future requests see them as occupied
for row in allocated.itertuples():

    occupied_intervals[row.allocated_room_id].append(
        (
            row.allocated_start_datetime,
            row.allocated_end_datetime
        )
    )

print(
    "capacity_feasible_alternative_count added:",
    candidate_dataset.shape
)

print(
    candidate_dataset[
        [
            "booking_id",
            "candidate_room_id",
            "candidate_capacity",
            "capacity_feasible_alternative_count"
        ]
    ].head(10)
)

capacity_feasible_alternative_count added: (61784, 89)
  booking_id candidate_room_id  candidate_capacity  \
0    RB06430              R008                  16   
1    RB10999              R042                   4   
2    RB10999              R045                   8   
3    RB19847              R034                  16   
4    RB22582              R024                   8   
5    RB22582              R025                   8   
6    RB22582              R030                   4   
7    RB25100              R041                  16   
8    RB26980              R043                   8   
9    RB26980              R047                   4   

   capacity_feasible_alternative_count  
0                                    6  
1                                   49  
2                                   27  
3                                    6  
4                                   27  
5                                   27  
6                                   49  
7                     

In [33]:
# ===================================================================================================================================
#                                                Current Request - Candidate Relationship 
# ====================================================================================================================================

# 1. --------------------------------------------       Capacity Difference    -----------------------------------------------------


candidate_dataset["capacity_difference"] = (
    candidate_dataset["booking_id"].map(
        room_booking_requests.set_index("booking_id")["requested_capacity"]
    )
    - candidate_dataset["candidate_capacity"]
).abs()

print(
    candidate_dataset[
        [
            "booking_id",
            "candidate_capacity",
            "capacity_difference"
        ]
    ].head(10)
)

print("Candidate dataset shape:", candidate_dataset.shape)


  booking_id  candidate_capacity  capacity_difference
0    RB06430                  16                    0
1    RB10999                   4                    0
2    RB10999                   8                    4
3    RB19847                  16                    4
4    RB22582                   8                    4
5    RB22582                   8                    4
6    RB22582                   4                    0
7    RB25100                  16                    6
8    RB26980                   8                    4
9    RB26980                   4                    0
Candidate dataset shape: (61784, 90)


In [34]:
# 2. --------------------------------------------       time_difference_minutes    -----------------------------------------------------

candidate_dataset["time_difference_minutes"] = (
    pd.to_datetime(candidate_dataset["candidate_start_datetime"])
    - pd.to_datetime(
        candidate_dataset["booking_id"].map(
            room_booking_requests.set_index("booking_id")[
                "requested_start_datetime"
            ]
        )
    )
).abs().dt.total_seconds() / 60

candidate_dataset["time_difference_minutes"] = (
    candidate_dataset["time_difference_minutes"].astype(int)
)

print(
    candidate_dataset[
        [
            "booking_id",
            "candidate_start_datetime",
            "time_difference_minutes"
        ]
    ].head(10)
)

print("Candidate dataset shape:", candidate_dataset.shape)

  booking_id candidate_start_datetime  time_difference_minutes
0    RB06430      2025-01-01 08:00:00                        0
1    RB10999      2025-01-01 08:00:00                        0
2    RB10999      2025-01-01 08:00:00                        0
3    RB19847      2025-01-01 08:00:00                        0
4    RB22582      2025-01-01 08:00:00                        0
5    RB22582      2025-01-01 08:00:00                        0
6    RB22582      2025-01-01 08:00:00                        0
7    RB25100      2025-01-01 08:00:00                        0
8    RB26980      2025-01-01 08:00:00                        0
9    RB26980      2025-01-01 08:00:00                        0
Candidate dataset shape: (61784, 91)


In [35]:
# =================================================================================================================================
#                                                    DEFINING THE TARGET (  Y  )
# ====================================================================================================================================

# Target component 1: employee's historical preference for the candidate room type

candidate_dataset["target_room_type_preference"] = (
    candidate_dataset["employee_candidate_type_pct"]
)

candidate_dataset[
    ["booking_id", "employee_id", "candidate_room_id",
     "employee_candidate_type_pct", "target_room_type_preference"]
].head(10)


,booking_id,employee_id,candidate_room_id,employee_candidate_type_pct,target_room_type_preference
0,RB06430,E0742,R008,0.0,0.0
1,RB10999,E0166,R042,0.0,0.0
2,RB10999,E0166,R045,0.0,0.0
3,RB19847,E0605,R034,0.0,0.0
4,RB22582,E0904,R024,0.0,0.0
5,RB22582,E0904,R025,0.0,0.0
6,RB22582,E0904,R030,0.0,0.0
7,RB25100,E0378,R041,0.0,0.0
8,RB26980,E0506,R043,0.0,0.0
9,RB26980,E0506,R047,0.0,0.0


In [36]:
# Target component 2: employee's historical preference for this specific candidate room

candidate_dataset["target_room_preference"] = (
    candidate_dataset["employee_candidate_room_pct"]
)

candidate_dataset[
    ["booking_id", "employee_id", "candidate_room_id",
     "employee_candidate_room_pct", "target_room_preference"]
].head(10)

,booking_id,employee_id,candidate_room_id,employee_candidate_room_pct,target_room_preference
0,RB06430,E0742,R008,0.0,0.0
1,RB10999,E0166,R042,0.0,0.0
2,RB10999,E0166,R045,0.0,0.0
3,RB19847,E0605,R034,0.0,0.0
4,RB22582,E0904,R024,0.0,0.0
5,RB22582,E0904,R025,0.0,0.0
6,RB22582,E0904,R030,0.0,0.0
7,RB25100,E0378,R041,0.0,0.0
8,RB26980,E0506,R043,0.0,0.0
9,RB26980,E0506,R047,0.0,0.0


In [40]:
# Target component 3: employee's historical preference for candidate floor
# Recent behaviour is weighted more than long-term behaviour.

candidate_dataset["target_floor_preference"] = candidate_dataset.apply(
    lambda row: (
        0.4 * row[f"floor_{int(row['candidate_floor'])}_pct_long_term"]
        + 0.6 * row[f"floor_{int(row['candidate_floor'])}_pct_recent"]
    ),
    axis=1
)

candidate_dataset[
    ["booking_id", "employee_id", "candidate_room_id",
     "candidate_floor", "target_floor_preference"]
].head(10)

,booking_id,employee_id,candidate_room_id,candidate_floor,target_floor_preference
0,RB06430,E0742,R008,1,0.000000
1,RB10999,E0166,R042,5,94.871795
2,RB10999,E0166,R045,5,94.871795
3,RB19847,E0605,R034,4,0.000000
4,RB22582,E0904,R024,3,60.810811
5,RB22582,E0904,R025,3,60.810811
6,RB22582,E0904,R030,3,60.810811
7,RB25100,E0378,R041,5,94.666667
8,RB26980,E0506,R043,5,91.428571
9,RB26980,E0506,R047,5,91.428571


In [39]:
[col for col in candidate_dataset.columns if "floor" in col.lower()]

['candidate_floor',
 'floor_1_pct_long_term',
 'floor_2_pct_long_term',
 'floor_3_pct_long_term',
 'floor_4_pct_long_term',
 'floor_5_pct_long_term',
 'floor_1_pct_recent',
 'floor_2_pct_recent',
 'floor_3_pct_recent',
 'floor_4_pct_recent',
 'floor_5_pct_recent',
 'floor_shift_pct',
 'employee_candidate_floor_count_x',
 'employee_candidate_floor_pct_x',
 'employee_candidate_floor_count_y',
 'employee_candidate_floor_pct_y',
 'employee_candidate_floor_count',
 'employee_candidate_floor_pct']

In [42]:
[col for col in candidate_dataset.columns if "hour" in col.lower()]

['start_hour_8_pct_long_term',
 'start_hour_9_pct_long_term',
 'start_hour_10_pct_long_term',
 'start_hour_11_pct_long_term',
 'start_hour_8_pct_recent',
 'start_hour_9_pct_recent',
 'start_hour_10_pct_recent',
 'start_hour_11_pct_recent',
 'start_hour_shift_pct',
 'employee_candidate_hour_count_x',
 'employee_candidate_hour_pct_x',
 'employee_candidate_hour_count_y',
 'employee_candidate_hour_pct_y',
 'employee_candidate_hour_count',
 'employee_candidate_hour_pct']

In [46]:
# Target component 4: employee's historical preference for candidate start hour

candidate_dataset["candidate_hour"] = (
    pd.to_datetime(candidate_dataset["candidate_start_datetime"]).dt.hour
)

def get_hour_preference(row):
    hour = int(row["candidate_hour"])

    if hour in [8, 9, 10, 11]:
        long_term = row[f"start_hour_{hour}_pct_long_term"]
        recent = row[f"start_hour_{hour}_pct_recent"]

        return 0.4 * long_term + 0.6 * recent

    return 0.0


candidate_dataset["target_hour_preference"] = candidate_dataset.apply(
    get_hour_preference,
    axis=1
)

candidate_dataset[
    ["booking_id", "employee_id", "candidate_room_id",
     "candidate_hour", "target_hour_preference"]
].head(10)

,booking_id,employee_id,candidate_room_id,candidate_hour,target_hour_preference
0,RB06430,E0742,R008,8,28.000000
1,RB10999,E0166,R042,8,11.282051
2,RB10999,E0166,R045,8,11.282051
3,RB19847,E0605,R034,8,5.714286
4,RB22582,E0904,R024,8,91.351351
5,RB22582,E0904,R025,8,91.351351
6,RB22582,E0904,R030,8,91.351351
7,RB25100,E0378,R041,8,50.666667
8,RB26980,E0506,R043,8,47.857143
9,RB26980,E0506,R047,8,47.857143


In [47]:
# Final candidate relevance target (y)

candidate_dataset["target"] = (
    0.30 * (candidate_dataset["target_room_type_preference"] / 100)
    + 0.30 * (candidate_dataset["target_room_preference"] / 100)
    + 0.20 * (candidate_dataset["target_floor_preference"] / 100)
    + 0.20 * (candidate_dataset["target_hour_preference"] / 100)
)

candidate_dataset[
    ["booking_id", "employee_id", "candidate_room_id",
     "target_room_type_preference",
     "target_room_preference",
     "target_floor_preference",
     "target_hour_preference",
     "target"]
].head(10)

,booking_id,employee_id,candidate_room_id,target_room_type_preference,target_room_preference,target_floor_preference,target_hour_preference,target
0,RB06430,E0742,R008,0.0,0.0,0.000000,28.000000,0.056000
1,RB10999,E0166,R042,0.0,0.0,94.871795,11.282051,0.212308
2,RB10999,E0166,R045,0.0,0.0,94.871795,11.282051,0.212308
3,RB19847,E0605,R034,0.0,0.0,0.000000,5.714286,0.011429
4,RB22582,E0904,R024,0.0,0.0,60.810811,91.351351,0.304324
5,RB22582,E0904,R025,0.0,0.0,60.810811,91.351351,0.304324
6,RB22582,E0904,R030,0.0,0.0,60.810811,91.351351,0.304324
7,RB25100,E0378,R041,0.0,0.0,94.666667,50.666667,0.290667
8,RB26980,E0506,R043,0.0,0.0,91.428571,47.857143,0.278571
9,RB26980,E0506,R047,0.0,0.0,91.428571,47.857143,0.278571


In [48]:
#  Again redesigning target architecture

# ============================================================
# FINAL TARGET (y): Employee-specific candidate relevance
# ============================================================

# Bring the employee's underlying preferences into candidate rows
employee_preferences = employees[
    [
        "employee_id",
        "preferred_room_type",
        "preferred_floor",
        "preferred_start_hour",
        "typical_capacity"
    ]
].drop_duplicates("employee_id")

candidate_dataset = candidate_dataset.merge(
    employee_preferences,
    on="employee_id",
    how="left",
    validate="many_to_one"
)

# 1. Room type preference
candidate_dataset["target_type_score"] = (
    candidate_dataset["candidate_room_type"]
    == candidate_dataset["preferred_room_type"]
).astype(float)

# 2. Floor preference
floor_distance = (
    candidate_dataset["candidate_floor"]
    - candidate_dataset["preferred_floor"]
).abs()

candidate_dataset["target_floor_score"] = np.select(
    [
        floor_distance == 0,
        floor_distance == 1,
        floor_distance == 2
    ],
    [
        1.0,
        0.75,
        0.50
    ],
    default=0.25
)

# 3. Start-hour preference
candidate_hour = pd.to_datetime(
    candidate_dataset["candidate_start_datetime"]
).dt.hour

hour_distance = (
    candidate_hour
    - candidate_dataset["preferred_start_hour"]
).abs()

candidate_dataset["target_time_score"] = np.select(
    [
        hour_distance == 0,
        hour_distance == 1,
        hour_distance == 2
    ],
    [
        1.0,
        0.75,
        0.50
    ],
    default=0.25
)

# 4. Capacity preference
capacity_distance = (
    candidate_dataset["candidate_capacity"]
    - candidate_dataset["typical_capacity"]
).abs()

candidate_dataset["target_capacity_score"] = np.select(
    [
        capacity_distance == 0,
        capacity_distance <= 2,
        capacity_distance <= 4
    ],
    [
        1.0,
        0.75,
        0.50
    ],
    default=0.25
)

# Final target y: 0 to 1
candidate_dataset["target"] = (
    0.35 * candidate_dataset["target_type_score"]
    + 0.25 * candidate_dataset["target_floor_score"]
    + 0.20 * candidate_dataset["target_time_score"]
    + 0.20 * candidate_dataset["target_capacity_score"]
)

candidate_dataset[
    [
        "booking_id",
        "employee_id",
        "candidate_room_id",
        "target_type_score",
        "target_floor_score",
        "target_time_score",
        "target_capacity_score",
        "target"
    ]
].head(10)

,booking_id,employee_id,candidate_room_id,target_type_score,target_floor_score,target_time_score,target_capacity_score,target
0,RB06430,E0742,R008,0.0,0.5,0.75,1.00,0.475
1,RB10999,E0166,R042,1.0,1.0,0.75,1.00,0.950
2,RB10999,E0166,R045,1.0,1.0,0.75,0.50,0.850
3,RB19847,E0605,R034,1.0,0.5,0.75,0.50,0.725
4,RB22582,E0904,R024,1.0,1.0,1.00,0.25,0.850
5,RB22582,E0904,R025,1.0,1.0,1.00,0.25,0.850
6,RB22582,E0904,R030,1.0,1.0,1.00,0.75,0.950
7,RB25100,E0378,R041,0.0,1.0,0.75,0.25,0.450
8,RB26980,E0506,R043,0.0,1.0,1.00,0.75,0.600
9,RB26980,E0506,R047,0.0,1.0,1.00,0.75,0.600


In [49]:
# ============================================================
# Prepare X (features) and y (target)
# ============================================================

# Columns used only for target construction / identification
drop_from_X = [
    "target",
    "target_type_score",
    "target_floor_score",
    "target_time_score",
    "target_capacity_score",
    "target_room_type_preference",
    "target_room_preference",
    "target_floor_preference",
    "target_hour_preference",
    
    # Ground-truth employee preferences used to create y
    "preferred_room_type",
    "preferred_floor",
    "preferred_start_hour",
    "typical_capacity",

    # Temporary helper
    "candidate_hour"
]

# Keep IDs separately for ranking groups and later evaluation
ranking_info = candidate_dataset[
    ["booking_id", "employee_id", "candidate_room_id"]
].copy()

# X = all remaining model features
X = candidate_dataset.drop(
    columns=drop_from_X + ["booking_id", "employee_id", "candidate_room_id"],
    errors="ignore"
)

# y = final relevance target
y = candidate_dataset["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target range:", y.min(), "to", y.max())

X shape: (61784, 88)
y shape: (61784,)
Target range: 0.21250000000000002 to 1.0


In [50]:
# Check feature data types before model training

print("Numeric features:", X.select_dtypes(include=["number"]).shape[1])
print("Categorical features:", X.select_dtypes(include=["object", "category"]).shape[1])
print("Datetime features:", X.select_dtypes(include=["datetime", "datetimetz"]).shape[1])

print("\nCategorical columns:")
print(X.select_dtypes(include=["object", "category"]).columns.tolist())

print("\nDatetime columns:")
print(X.select_dtypes(include=["datetime", "datetimetz"]).columns.tolist())

Numeric features: 84
Categorical features: 2
Datetime features: 2

Categorical columns:
['candidate_room_type', 'allocation_rule']

Datetime columns:
['candidate_start_datetime', 'candidate_end_datetime']


In [51]:
# Convert candidate datetime information into model-friendly numeric features

candidate_start = pd.to_datetime(X["candidate_start_datetime"])
candidate_end = pd.to_datetime(X["candidate_end_datetime"])

X["candidate_start_hour"] = candidate_start.dt.hour
X["candidate_start_minute"] = candidate_start.dt.minute
X["candidate_duration_minutes"] = (
    (candidate_end - candidate_start).dt.total_seconds() / 60
)

# Original datetime columns are not needed by the model
X = X.drop(
    columns=["candidate_start_datetime", "candidate_end_datetime"]
)

print("X shape:", X.shape)
print("Remaining non-numeric columns:")
print(X.select_dtypes(exclude=["number"]).columns.tolist())

X shape: (61784, 89)
Remaining non-numeric columns:
['candidate_room_type', 'allocation_rule']


In [52]:
# ============================================================
# Chronological train / validation / test split
# Same booking's candidates always stay together
# ============================================================

booking_dates = (
    room_booking_requests[
        ["booking_id", "requested_start_datetime"]
    ]
    .drop_duplicates("booking_id")
    .sort_values("requested_start_datetime")
    .reset_index(drop=True)
)

n_bookings = len(booking_dates)

train_end = int(n_bookings * 0.70)
val_end = int(n_bookings * 0.85)

train_bookings = set(
    booking_dates.iloc[:train_end]["booking_id"]
)

val_bookings = set(
    booking_dates.iloc[train_end:val_end]["booking_id"]
)

test_bookings = set(
    booking_dates.iloc[val_end:]["booking_id"]
)

train_mask = ranking_info["booking_id"].isin(train_bookings)
val_mask = ranking_info["booking_id"].isin(val_bookings)
test_mask = ranking_info["booking_id"].isin(test_bookings)

X_train = X.loc[train_mask].copy()
X_val = X.loc[val_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_val = y.loc[val_mask].copy()
y_test = y.loc[test_mask].copy()

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (43329, 89) (43329,)
Validation: (9141, 89) (9141,)
Test: (9314, 89) (9314,)


In [54]:
%pip install catboost

   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.3/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/100.2 MB 718.9 kB/s eta 0:02:19
   ---------------------------------------- 0.8/100.2 MB 971.1 kB/s eta 0:01:43
   ---------------------------------------- 1.0/100.2 MB 1.1 MB/s eta 0:01:34
    --------------------------------------- 1.3/100.2 MB 1.0 MB/s eta 0:01:37
    --------------------------------------- 1.6/100.2 MB 1.1 MB/s eta 0:01:32
    --------------------------------------- 2.1/100.2 MB 1.2 MB/s eta 0:01:21
    --------------------------------------- 2.4/100.2 MB 1.3 MB/s eta 0:01:16
   - -------------------------------------- 2.9/100.2 MB 1.4 MB/s eta 0:01:12
   - -------------------------------------- 3.1/100.2 MB 1.4 MB/s eta 0:01:10
   - -------------------------------------- 3.7/100.2 MB 1.5 MB/s eta 0:01:06
   


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [55]:
# Prepare categorical features for CatBoost ranking

categorical_features = [
    "candidate_room_type",
    "allocation_rule"
]

for col in categorical_features:
    X_train[col] = X_train[col].astype(str)
    X_val[col] = X_val[col].astype(str)
    X_test[col] = X_test[col].astype(str)

cat_feature_indices = [
    X_train.columns.get_loc(col)
    for col in categorical_features
]

print("Categorical feature indices:", cat_feature_indices)
print("Categorical features:", categorical_features)

Categorical feature indices: [0, 3]
Categorical features: ['candidate_room_type', 'allocation_rule']


In [56]:
# Ranking group IDs for CatBoost
# Each booking_id represents one candidate-ranking problem

group_train = ranking_info.loc[
    train_mask, "booking_id"
].values

group_val = ranking_info.loc[
    val_mask, "booking_id"
].values

group_test = ranking_info.loc[
    test_mask, "booking_id"
].values

print("Train groups:", len(np.unique(group_train)))
print("Validation groups:", len(np.unique(group_val)))
print("Test groups:", len(np.unique(group_test)))

Train groups: 22773
Validation groups: 4786
Test groups: 4883


In [57]:
# Keep all candidates of the same booking together
# This is required for proper ranking-group training.

train_order = ranking_info.loc[train_mask].sort_values("booking_id").index
val_order = ranking_info.loc[val_mask].sort_values("booking_id").index
test_order = ranking_info.loc[test_mask].sort_values("booking_id").index

X_train = X_train.loc[train_order]
y_train = y_train.loc[train_order]

X_val = X_val.loc[val_order]
y_val = y_val.loc[val_order]

X_test = X_test.loc[test_order]
y_test = y_test.loc[test_order]

group_train = ranking_info.loc[train_order, "booking_id"].values
group_val = ranking_info.loc[val_order, "booking_id"].values
group_test = ranking_info.loc[test_order, "booking_id"].values

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))
print("First train groups:", group_train[:10])

Training rows: 43329
Validation rows: 9141
Test rows: 9314
First train groups: ['RB00001' 'RB00001' 'RB00002' 'RB00002' 'RB00003' 'RB00004' 'RB00008'
 'RB00009' 'RB00009' 'RB00011']


In [58]:
from catboost import CatBoostRanker, Pool

# Create CatBoost ranking pools
train_pool = Pool(
    data=X_train,
    label=y_train,
    group_id=group_train,
    cat_features=cat_feature_indices
)

val_pool = Pool(
    data=X_val,
    label=y_val,
    group_id=group_val,
    cat_features=cat_feature_indices
)

# Ranking model
ranker = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=3",
    iterations=500,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    random_strength=1,
    l2_leaf_reg=5,
    verbose=50,
    od_type="Iter",
    od_wait=50,
    use_best_model=True
)

# Train
ranker.fit(
    train_pool,
    eval_set=val_pool
)

print("\nRanking model training complete.")

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.9964996	best: 0.9964996 (0)	total: 246ms	remaining: 2m 2s
50:	test: 0.9984871	best: 0.9985212 (47)	total: 3.35s	remaining: 29.5s
100:	test: 0.9988312	best: 0.9988312 (99)	total: 6.29s	remaining: 24.9s
150:	test: 0.9989187	best: 0.9989187 (150)	total: 9.08s	remaining: 21s
200:	test: 0.9990033	best: 0.9990033 (199)	total: 11.9s	remaining: 17.7s
250:	test: 0.9990620	best: 0.9990653 (234)	total: 14.6s	remaining: 14.4s
300:	test: 0.9991737	best: 0.9991737 (300)	total: 17.2s	remaining: 11.4s
350:	test: 0.9992497	best: 0.9992788 (339)	total: 19.8s	remaining: 8.4s
400:	test: 0.9992823	best: 0.9992886 (385)	total: 22.7s	remaining: 5.6s
450:	test: 0.9992923	best: 0.9993021 (416)	total: 25.9s	remaining: 2.81s
499:	test: 0.9993377	best: 0.9993460 (487)	total: 29.1s	remaining: 0us

bestTest = 0.9993459942
bestIteration = 487

Shrink model to first 488 iterations.

Ranking model training complete.


In [59]:
from sklearn.metrics import ndcg_score

# Predict scores for unseen test candidates
test_predictions = ranker.predict(X_test)

test_results = ranking_info.loc[test_mask].copy()
test_results["actual_target"] = y_test.values
test_results["predicted_score"] = test_predictions

# Rank candidates within each booking
test_results["predicted_rank"] = (
    test_results.groupby("booking_id")["predicted_score"]
    .rank(method="first", ascending=False)
)

# Actual best candidate according to target
test_results["actual_best"] = (
    test_results["actual_target"]
    == test_results.groupby("booking_id")["actual_target"].transform("max")
)

# Did model put an actual-best candidate at rank 1?
top1_accuracy = (
    test_results[
        (test_results["predicted_rank"] == 1)
        & (test_results["actual_best"])
    ]
    .groupby("booking_id")
    .size()
    .shape[0]
    / test_results["booking_id"].nunique()
)

print("Test booking groups:", test_results["booking_id"].nunique())
print(f"Top-1 ranking accuracy: {top1_accuracy:.4f}")
print(f"Top-1 ranking accuracy: {top1_accuracy * 100:.2f}%")

print("\nSample rankings:")
print(
    test_results
    .sort_values(["booking_id", "predicted_rank"])
    [
        ["booking_id", "candidate_room_id",
         "actual_target", "predicted_score", "predicted_rank"]
    ]
    .head(20)
)

Test booking groups: 4883
Top-1 ranking accuracy: 0.9017
Top-1 ranking accuracy: 90.17%

Sample rankings:
      booking_id candidate_room_id  actual_target  predicted_score  \
52812    RB00010              R008         0.9500         1.953301   
57825    RB00014              R046         0.5000         0.136494   
58054    RB00017              R002         0.8875         2.076872   
58433    RB00019              R012         0.4750        -1.309566   
58799    RB00034              R003         0.9375         1.713927   
60073    RB00052              R030         0.4250         0.265876   
60071    RB00052              R020         0.3250        -0.378545   
60072    RB00052              R022         0.7750        -0.458174   
57160    RB00061              R003         0.9000         2.226614   
57161    RB00061              R004         0.5500        -0.494669   
57162    RB00061              R008         0.8000        -1.580067   
57200    RB00066              R045         0.8500     

In [62]:
# ============================================================
# Test Ranking Metrics: Top-3 Hit Rate + NDCG@3
# Skip single-candidate groups for NDCG
# ============================================================

top3_hits = []
ndcg_scores = []

for booking_id, group in test_results.groupby("booking_id"):
    actual = group["actual_target"].values
    predicted = group["predicted_score"].values

    # Top-3 predicted candidates
    top3_indices = np.argsort(predicted)[::-1][:3]

    # Highest-target candidate inside model's Top-3?
    best_actual = actual.max()
    top3_actual = actual[top3_indices]

    top3_hits.append(best_actual in top3_actual)

    # NDCG requires at least 2 candidates
    if len(group) > 1:
        k = min(3, len(group))

        ndcg = ndcg_score(
            [actual],
            [predicted],
            k=k
        )

        ndcg_scores.append(ndcg)

top3_hit_rate = np.mean(top3_hits)
mean_ndcg_3 = np.mean(ndcg_scores)

print(f"Test booking groups : {test_results['booking_id'].nunique()}")
print(f"NDCG evaluated groups: {len(ndcg_scores)}")

print(f"\nTop-1 Accuracy : {top1_accuracy:.4f} ({top1_accuracy * 100:.2f}%)")
print(f"Top-3 Hit Rate  : {top3_hit_rate:.4f} ({top3_hit_rate * 100:.2f}%)")
print(f"Mean NDCG@3     : {mean_ndcg_3:.4f}")

Test booking groups : 4883
NDCG evaluated groups: 2082

Top-1 Accuracy : 0.9017 (90.17%)
Top-3 Hit Rate  : 0.9855 (98.55%)
Mean NDCG@3     : 0.9770


In [63]:
#  Now check feature importance
# ============================================================
# Feature importance from the trained ranking model
# ============================================================

feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": ranker.get_feature_importance(train_pool)
})

feature_importance = (
    feature_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 20 features:")
print(feature_importance.head(20).to_string(index=False))



Top 20 features:
                            feature  importance
                    candidate_floor    0.083226
                capacity_difference    0.069091
              floor_1_pct_long_term    0.015255
              floor_4_pct_long_term    0.011851
              floor_5_pct_long_term    0.009387
              floor_2_pct_long_term    0.008544
                    allocation_rule    0.005989
      employee_candidate_type_pct_y    0.005825
      employee_candidate_type_pct_x    0.005701
              room_utilization_rate    0.005632
                candidate_room_type    0.005012
          capacity_10_pct_long_term    0.004543
        employee_candidate_type_pct    0.004404
           capacity_4_pct_long_term    0.003357
           capacity_8_pct_long_term    0.003207
           capacity_2_pct_long_term    0.002754
                 candidate_capacity    0.002148
capacity_feasible_alternative_count    0.002077
                  Meeting_long_term    0.001434
     employee_candidate

In [64]:
# ============================================================
# Historical Allocator vs ML Ranker
# Compare employee-relevance of the selected candidate
# ============================================================

# Historical allocator's actual room for each booking
historical_choice = (
    room_bookings[
        room_bookings["booking_status"] == "Allocated"
    ][
        ["booking_id", "allocated_room_id"]
    ]
    .drop_duplicates("booking_id")
    .rename(columns={"allocated_room_id": "allocator_room_id"})
)

# Test results with candidate target scores
comparison = test_results.merge(
    historical_choice,
    on="booking_id",
    how="left"
)

# Target relevance of the historical allocator's selected room
comparison["allocator_target"] = np.where(
    comparison["candidate_room_id"] == comparison["allocator_room_id"],
    comparison["actual_target"],
    np.nan
)

allocator_scores = (
    comparison
    .groupby("booking_id")["allocator_target"]
    .max()
)

# ML's #1 candidate target relevance
ml_scores = (
    comparison[
        comparison["predicted_rank"] == 1
    ]
    .set_index("booking_id")["actual_target"]
)

# Keep only bookings where both are available
baseline_comparison = pd.DataFrame({
    "allocator_target": allocator_scores,
    "ml_target": ml_scores
}).dropna()

baseline_comparison["improvement"] = (
    baseline_comparison["ml_target"]
    - baseline_comparison["allocator_target"]
)

print("Bookings compared:", len(baseline_comparison))

print(
    f"\nAllocator mean target: "
    f"{baseline_comparison['allocator_target'].mean():.4f}"
)

print(
    f"ML mean target: "
    f"{baseline_comparison['ml_target'].mean():.4f}"
)

print(
    f"Mean improvement: "
    f"{baseline_comparison['improvement'].mean():+.4f}"
)

print(
    f"ML better: "
    f"{(baseline_comparison['improvement'] > 0).mean() * 100:.2f}%"
)

print(
    f"Same: "
    f"{(baseline_comparison['improvement'] == 0).mean() * 100:.2f}%"
)

print(
    f"Allocator better: "
    f"{(baseline_comparison['improvement'] < 0).mean() * 100:.2f}%"
)

Bookings compared: 4883

Allocator mean target: 0.6453
ML mean target: 0.6773
Mean improvement: +0.0320
ML better: 16.67%
Same: 78.99%
Allocator better: 4.34%


In [65]:
# ============================================================
# EXPERIMENT A: Personalization-only Ranker
# Remove scarcity features, keep everything else identical
# ============================================================

scarcity_features = [
    "room_historical_demand",
    "room_time_demand",
    "room_utilization_rate",
    "capacity_feasible_alternative_count"
]

X_train_personal = X_train.drop(columns=scarcity_features)
X_val_personal = X_val.drop(columns=scarcity_features)
X_test_personal = X_test.drop(columns=scarcity_features)

# Categorical column positions change after dropping columns
cat_features_personal = [
    X_train_personal.columns.get_loc(col)
    for col in categorical_features
]

train_pool_personal = Pool(
    data=X_train_personal,
    label=y_train,
    group_id=group_train,
    cat_features=cat_features_personal
)

val_pool_personal = Pool(
    data=X_val_personal,
    label=y_val,
    group_id=group_val,
    cat_features=cat_features_personal
)

personal_ranker = CatBoostRanker(
    loss_function="YetiRank",
    eval_metric="NDCG:top=3",
    iterations=500,
    learning_rate=0.05,
    depth=8,
    random_seed=42,
    random_strength=1,
    l2_leaf_reg=5,
    verbose=100,
    od_type="Iter",
    od_wait=50,
    use_best_model=True
)

personal_ranker.fit(
    train_pool_personal,
    eval_set=val_pool_personal
)

print("\nPersonalization-only model trained.")

Groupwise loss function. OneHotMaxSize set to 10
0:	test: 0.9964534	best: 0.9964534 (0)	total: 74ms	remaining: 36.9s
100:	test: 0.9988308	best: 0.9988308 (100)	total: 10.4s	remaining: 41.2s
200:	test: 0.9990073	best: 0.9990073 (198)	total: 22.5s	remaining: 33.5s
300:	test: 0.9992434	best: 0.9992434 (300)	total: 27.7s	remaining: 18.3s
400:	test: 0.9993168	best: 0.9993264 (369)	total: 32.7s	remaining: 8.07s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9993263956
bestIteration = 369

Shrink model to first 370 iterations.

Personalization-only model trained.


In [66]:
# ============================================================
# Evaluate Personalization-only model on the same test set
# ============================================================

personal_predictions = personal_ranker.predict(X_test_personal)

personal_results = ranking_info.loc[test_mask].copy()
personal_results["actual_target"] = y_test.values
personal_results["predicted_score"] = personal_predictions

# Rank within each booking
personal_results["predicted_rank"] = (
    personal_results
    .groupby("booking_id")["predicted_score"]
    .rank(method="first", ascending=False)
)

# Top-1
personal_results["actual_best"] = (
    personal_results["actual_target"]
    == personal_results.groupby("booking_id")["actual_target"].transform("max")
)

personal_top1 = (
    personal_results[
        (personal_results["predicted_rank"] == 1)
        & personal_results["actual_best"]
    ]
    .groupby("booking_id")
    .size()
    .shape[0]
    / personal_results["booking_id"].nunique()
)

# Top-3 + NDCG@3
personal_top3_hits = []
personal_ndcg = []

for booking_id, group in personal_results.groupby("booking_id"):
    actual = group["actual_target"].values
    predicted = group["predicted_score"].values

    top3_indices = np.argsort(predicted)[::-1][:3]

    personal_top3_hits.append(
        actual.max() in actual[top3_indices]
    )

    if len(group) > 1:
        k = min(3, len(group))
        personal_ndcg.append(
            ndcg_score([actual], [predicted], k=k)
        )

personal_top3 = np.mean(personal_top3_hits)
personal_ndcg3 = np.mean(personal_ndcg)
personal_mean_target = (
    personal_results[
        personal_results["predicted_rank"] == 1
    ]["actual_target"].mean()
)

print(f"Personalization-only Top-1 : {personal_top1:.4f} ({personal_top1*100:.2f}%)")
print(f"Personalization-only Top-3 : {personal_top3:.4f} ({personal_top3*100:.2f}%)")
print(f"Personalization-only NDCG@3: {personal_ndcg3:.4f}")
print(f"Personalization-only Mean Target: {personal_mean_target:.4f}")

Personalization-only Top-1 : 0.9001 (90.01%)
Personalization-only Top-3 : 0.9844 (98.44%)
Personalization-only NDCG@3: 0.9766
Personalization-only Mean Target: 0.6767


In [67]:
# Business impact: booking success rate before vs after ML

total_requests = len(room_booking_requests)

allocator_allocated = (
    room_bookings[
        room_bookings["booking_status"] == "Allocated"
    ]["booking_id"]
    .nunique()
)

# ML can only rank requests that already have feasible candidates.
# Therefore every request with >=1 candidate remains allocatable.
ml_allocated = candidate_dataset["booking_id"].nunique()

allocator_rate = allocator_allocated / total_requests * 100
ml_rate = ml_allocated / total_requests * 100

print(f"Total requests: {total_requests}")
print(f"Allocator successful bookings: {allocator_allocated}")
print(f"ML successful bookings: {ml_allocated}")

print(f"\nAllocator success rate: {allocator_rate:.2f}%")
print(f"ML pipeline success rate: {ml_rate:.2f}%")
print(f"Change: {ml_rate - allocator_rate:+.2f} percentage points")

Total requests: 40000
Allocator successful bookings: 32442
ML successful bookings: 32442

Allocator success rate: 81.11%
ML pipeline success rate: 81.11%
Change: +0.00 percentage points


In [68]:
# Save the final trained ranking model

import os

os.makedirs("../models", exist_ok=True)

ranker.save_model("../models/room_ranking_model.cbm")

print("Model saved successfully.")
print("../models/room_ranking_model.cbm")

Model saved successfully.
../models/room_ranking_model.cbm


In [69]:
# Verify that the saved model can be loaded successfully

from catboost import CatBoostRanker

loaded_ranker = CatBoostRanker()
loaded_ranker.load_model("../models/room_ranking_model.cbm")

print("Saved model loaded successfully.")
print("Number of trees:", loaded_ranker.tree_count_)
print("Number of features:", len(loaded_ranker.feature_names_))

Saved model loaded successfully.
Number of trees: 488
Number of features: 89


In [70]:
# Save the final feature schema
import os
import json

os.makedirs("../models", exist_ok=True)

feature_schema = {
    "features": list(X.columns),
    "categorical_features": ["candidate_room_type", "allocation_rule"]
}

with open("../models/feature_schema.json", "w") as f:
    json.dump(feature_schema, f, indent=2)

print("Feature schema saved successfully.")
print("Number of features:", len(feature_schema["features"]))
print("../models/feature_schema.json")

Feature schema saved successfully.
Number of features: 89
../models/feature_schema.json


In [72]:
# Load final behaviour tables and create one deployment lookup

import os
import pandas as pd

os.makedirs("../models", exist_ok=True)

room_type_comparison = pd.read_csv(
    "../data/processed/room_type_behavior_features.csv"
)
floor_comparison = pd.read_csv(
    "../data/processed/floor_behavior_features.csv"
)
capacity_comparison = pd.read_csv(
    "../data/processed/capacity_behavior_features.csv"
)
start_hour_comparison = pd.read_csv(
    "../data/processed/start_hour_behavior_features.csv"
)

employee_behavior_lookup = (
    room_type_comparison
    .merge(floor_comparison, on="employee_id", how="outer")
    .merge(capacity_comparison, on="employee_id", how="outer")
    .merge(start_hour_comparison, on="employee_id", how="outer")
)

employee_behavior_lookup.to_csv(
    "../models/employee_behavior_lookup.csv",
    index=False
)

print("Employee behaviour lookup saved successfully.")
print("Employees:", employee_behavior_lookup["employee_id"].nunique())
print("../models/employee_behavior_lookup.csv")

Employee behaviour lookup saved successfully.
Employees: 880
../models/employee_behavior_lookup.csv


In [75]:
# Save room-level scarcity lookup tables for deployment

import os
import pandas as pd

os.makedirs("../models", exist_ok=True)

# Create hour directly from candidate start datetime
candidate_dataset["candidate_start_hour"] = (
    pd.to_datetime(candidate_dataset["candidate_start_datetime"]).dt.hour
)

# Latest historical room-level scarcity state
room_scarcity_lookup = (
    candidate_dataset
    .sort_values("booking_id")
    .groupby("candidate_room_id", as_index=False)
    .agg(
        room_historical_demand=("room_historical_demand", "last"),
        room_utilization_rate=("room_utilization_rate", "last")
    )
)

# Room + hour scarcity state
room_time_scarcity_lookup = (
    candidate_dataset
    .sort_values("booking_id")
    .groupby(
        ["candidate_room_id", "candidate_start_hour"],
        as_index=False
    )
    .agg(
        room_time_demand=("room_time_demand", "last")
    )
)

room_scarcity_lookup.to_csv(
    "../models/room_scarcity_lookup.csv",
    index=False
)

room_time_scarcity_lookup.to_csv(
    "../models/room_time_scarcity_lookup.csv",
    index=False
)

print("Scarcity lookup tables saved successfully.")
print("Rooms:", room_scarcity_lookup["candidate_room_id"].nunique())
print("Room-hour rows:", len(room_time_scarcity_lookup))

Scarcity lookup tables saved successfully.
Rooms: 50
Room-hour rows: 214


In [76]:
# Reusable ML ranking function for deployment

import json
import pandas as pd
from catboost import CatBoostRanker

# Load saved model
deployment_model = CatBoostRanker()
deployment_model.load_model("../models/room_ranking_model.cbm")

# Load feature schema
with open("../models/feature_schema.json", "r") as f:
    deployment_schema = json.load(f)

FEATURES = deployment_schema["features"]


def rank_feasible_candidates(candidate_features):
    """
    Rank already-feasible room candidates and return the best one.
    """

    # Keep exactly the feature order used during training
    model_input = candidate_features[FEATURES].copy()

    # Generate ML ranking scores
    scores = deployment_model.predict(model_input)

    result = candidate_features.copy()
    result["ml_score"] = scores

    # Highest score = best candidate
    result = result.sort_values(
        "ml_score",
        ascending=False
    ).reset_index(drop=True)

    return result


print("Inference function ready.")
print("Expected features:", len(FEATURES))

Inference function ready.
Expected features: 89


In [78]:
# Recreate the two datetime-derived features used during model training

candidate_dataset["candidate_start_datetime"] = pd.to_datetime(
    candidate_dataset["candidate_start_datetime"]
)

candidate_dataset["candidate_end_datetime"] = pd.to_datetime(
    candidate_dataset["candidate_end_datetime"]
)

candidate_dataset["candidate_start_hour"] = (
    candidate_dataset["candidate_start_datetime"].dt.hour
)

candidate_dataset["candidate_start_minute"] = (
    candidate_dataset["candidate_start_datetime"].dt.minute
)

candidate_dataset["candidate_duration_minutes"] = (
    (
        candidate_dataset["candidate_end_datetime"]
        - candidate_dataset["candidate_start_datetime"]
    )
    .dt.total_seconds()
    / 60
).astype(int)

print("Missing deployment features added.")
print("candidate_start_hour:", candidate_dataset["candidate_start_hour"].notna().all())
print("candidate_start_minute:", candidate_dataset["candidate_start_minute"].notna().all())
print("candidate_duration_minutes:", candidate_dataset["candidate_duration_minutes"].notna().all())

Missing deployment features added.
candidate_start_hour: True
candidate_start_minute: True
candidate_duration_minutes: True


In [79]:
# ======================================================================================================================================
#                             Ab next hum allocator + ML ko connect karenge, yani actual end-to-end prediction.
# ===========================================================================================================================================

# Test ML ranking on one real multi-candidate booking

test_booking_id = (
    candidate_dataset["booking_id"]
    .value_counts()
    .loc[lambda x: x > 1]
    .index[0]
)

test_candidates = candidate_dataset[
    candidate_dataset["booking_id"] == test_booking_id
].copy()

ranked_candidates = rank_feasible_candidates(test_candidates)

print("Booking ID:", test_booking_id)
print("Number of feasible candidates:", len(ranked_candidates))
print("\nML-ranked candidates:")
print(
    ranked_candidates[
        [
            "booking_id",
            "candidate_room_id",
            "candidate_room_type",
            "candidate_floor",
            "candidate_start_datetime",
            "allocation_rule",
            "ml_score"
        ]
    ].to_string(index=False)
)

print("\nBest candidate:")
print(ranked_candidates.iloc[0]["candidate_room_id"])



Booking ID: RB06501
Number of feasible candidates: 22

ML-ranked candidates:
booking_id candidate_room_id candidate_room_type  candidate_floor candidate_start_datetime        allocation_rule  ml_score
   RB06501              R013          Conference                2      2025-11-14 09:00:00 similar_type_any_floor  1.053657
   RB06501              R016               Focus                2      2025-11-14 09:00:00 similar_type_any_floor  0.984243
   RB06501              R019               Focus                2      2025-11-14 09:00:00 similar_type_any_floor  0.924123
   RB06501              R014          Conference                2      2025-11-14 09:00:00 similar_type_any_floor  0.812977
   RB06501              R026            Training                3      2025-11-14 09:00:00 similar_type_any_floor  0.635180
   RB06501              R022               Focus                3      2025-11-14 09:00:00 similar_type_any_floor  0.569783
   RB06501              R029            Training       

In [80]:
# ======================================================================================================================================
#                                                      🔄 Allocator + ML integration
# ===========================================================================================================================================

# Core allocation decision layer: Deterministic feasibility + ML ranking

def allocate_with_ml(feasible_candidates):
    """
    Final decision layer after deterministic feasibility filtering.
    """

    if feasible_candidates is None or len(feasible_candidates) == 0:
        return None, "skipped_no_feasible_candidate"

    # Only one feasible candidate: no need for ML
    if len(feasible_candidates) == 1:
        selected = feasible_candidates.iloc[0].copy()
        selected["ml_score"] = None
        return selected, "direct_allocation"

    # Multiple feasible candidates: let ML rank them
    ranked = rank_feasible_candidates(feasible_candidates)

    selected = ranked.iloc[0].copy()

    return selected, "ml_ranked_allocation"


# Test using the real 22-candidate example from above
selected_candidate, decision = allocate_with_ml(test_candidates)

print("Booking ID:", selected_candidate["booking_id"])
print("Decision:", decision)
print("Selected room:", selected_candidate["candidate_room_id"])
print("ML score:", selected_candidate["ml_score"])

Booking ID: RB06501
Decision: ml_ranked_allocation
Selected room: R013
ML score: 1.053657106430419
